## Deo 1: ucitavanje podataka

Sirovi skup `K8.data` se nalazi unutar arhive `data/raw/p53+mutants.zip` (ugnežden u `p53_old_2010.zip`). Svaki red ima 5408 numeričkih atributa, zatim ciljnu klasu, pa prateći zarez koji pravi praznu kolonu na kraju. Nedostajuće vrednosti su označene znakom `?`.

In [1]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("data/raw")
ZIP_PATH = RAW_DIR / "p53+mutants.zip"
DATA_PATH = RAW_DIR / "K8.data"
N_FEATURES = 5408

In [2]:
# Extract K8.data from the nested archive on first run only.
if not DATA_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as outer:
        old = zipfile.ZipFile(io.BytesIO(outer.read("p53_old_2010.zip")))
        with old.open("p53_old_2010/K8.data") as src, open(DATA_PATH, "wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                dst.write(chunk)
    print("Extracted:", DATA_PATH)
else:
    print("Already extracted:", DATA_PATH)

Already extracted: data/raw/K8.data


In [3]:
# Features as float32 to keep memory low, class as string, drop trailing empty column.
dtypes = {i: "float32" for i in range(N_FEATURES)}
dtypes[N_FEATURES] = "object"
dtypes[N_FEATURES + 1] = "object"

df = pd.read_csv(DATA_PATH, header=None, na_values="?", dtype=dtypes)
df = df.drop(columns=[N_FEATURES + 1])

X = df.iloc[:, :N_FEATURES]
y = df.iloc[:, N_FEATURES].rename("class")
del df

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (16772, 5408)
y shape: (16772,)


In [4]:
# Class distribution and a quick look at missing values.
print("Class distribution:")
print(y.value_counts())
print("\nActive ratio: {:.2%}".format((y == "active").mean()))

missing_per_row = X.isna().sum(axis=1)
print("\nRows with missing values:", int((missing_per_row > 0).sum()))
print("Total missing cells:", int(X.isna().sum().sum()))

Class distribution:
class
inactive    16629
active        143
Name: count, dtype: int64

Active ratio: 0.85%

Rows with missing values: 180
Total missing cells: 901854
